# Lecture 13: Practice Exercises
### NLP Course 2027 — Week 07

---

These exercises accompany **Lecture 13: Fine-Tuning Transformers**.
Complete each exercise in the provided code cell.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate

MODEL_NAME = 'distilbert-base-uncased'

---

## Exercise 13.1

Fine-tune DistilBERT on the **IMDb** movie review dataset (50k reviews). Use `load_dataset('imdb')`. Compare to SST-2 accuracy.

In [ ]:
# Exercise 13.1
# YOUR CODE: load IMDb, tokenize, fine-tune DistilBERT, evaluate
# IMDb has 'text' and 'label' columns (0=neg, 1=pos)

# dataset = load_dataset('imdb')
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# ... tokenize, set up Trainer, train, evaluate ...
# Report final accuracy and compare with SST-2 (~91%)


---

## Exercise 13.2

Fine-tune for a **multi-class** problem. Use `load_dataset('ag_news')` (4 topics: World, Sports, Business, Sci/Tech). Set `num_labels=4`.

In [ ]:
# Exercise 13.2
# YOUR CODE: load AG News, fine-tune DistilBERT with num_labels=4
# AG News: 'text' column, 'label' column (0=World, 1=Sports, 2=Business, 3=Sci/Tech)

# dataset = load_dataset('ag_news')
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)
# ... train and evaluate ...


---

## Exercise 13.3

Implement **early stopping** using `EarlyStoppingCallback` from Hugging Face. How many epochs does training need?

In [ ]:
# Exercise 13.3
# YOUR CODE: add EarlyStoppingCallback to a Trainer
# Set num_train_epochs=10 and let early stopping decide when to stop

training_args = TrainingArguments(
    output_dir='./results/early_stopping',
    num_train_epochs=10,
    per_device_train_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    # YOUR CODE: set other args
)

# trainer = Trainer(
#     ...
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
# )


---

## Exercise 13.4

Compare fine-tuning (all weights updated) vs feature extraction (frozen BERT, only head trained). Which achieves better validation accuracy on the same dataset?

In [ ]:
# Exercise 13.4
from transformers import AutoModelForSequenceClassification

def get_model_feature_extraction(model_name, num_labels):
    """Load model with all BERT weights frozen; only train the classification head."""
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    # YOUR CODE: freeze all parameters except the classifier head
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params (feature extraction): {trainable:,}')
    return model

def get_model_fine_tuning(model_name, num_labels):
    """Load model for full fine-tuning (all weights trainable)."""
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params (fine-tuning): {trainable:,}')
    return model

# YOUR CODE: train both models on the same dataset, compare val accuracy


---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**